# RG3 gas: supervised geometry ensembles\nClass-conditional geometry, local neighbors, and RBF boundary ensembles.\nOuter observation-block holdouts; thresholds selected only inside training groups.\nOR/AND combinations are predeclared diagnostic candidates, not an independently validated final selection.\nOnly RH normal/gas; same 583 shots, 17 gas. No timestamps as predictors.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.spatial.distance import cdist
from sklearn.svm import SVC
from sklearn.covariance import LedoitWolf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix, average_precision_score
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
# New experiments: supervised boundary ensemble, local gas-vs-normal proximity,
# and class-conditional covariance ensemble. Never fit on outer test labels.
memo={}
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in memo: return memo[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    names=[v+'_'+m for v in ['shake','relation','combined'] for m in ['svm','neighbors','gaussian','three_mean']]
    if len(np.unique(y[ref]))<2:
        return {n:np.zeros(len(q)) for n in names}
    rr=np.full((len(ref),4),np.nan)
    for b in groups.iloc[ref].unique():
        mask=groups.iloc[ref].eq(b).to_numpy()
        if (~mask).any() and (y[ref[~mask]]==0).any():
            rr[mask]=res_features(ref[~mask],ref[mask])
    rq=res_features(ref,q)
    views={'shake':(d.iloc[ref][stdcols].to_numpy(float),d.iloc[q][stdcols].to_numpy(float)),
           'relation':(rr,rq)}
    views['combined']=(np.column_stack([views['shake'][0],rr]),np.column_stack([views['shake'][1],rq]))
    out={}
    for v,(tr,te) in views.items():
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        tr=imp.fit_transform(tr); te=imp.transform(te)
        sc=RobustScaler().fit(tr[y[ref]==0])
        tr=np.clip(sc.transform(tr),-30,30);te=np.clip(sc.transform(te),-30,30)
        normal=tr[y[ref]==0]; gas=tr[y[ref]==1]
        s=[]
        for C in [.1,1.,10.]:
            m=SVC(C=C,kernel='rbf',gamma='scale',class_weight='balanced')
            m.fit(tr,y[ref]);s.append(expit(m.decision_function(te)))
        out[v+'_svm']=np.mean(s,axis=0)
        dn=np.sort(cdist(te,normal),axis=1)
        dg=np.sort(cdist(te,gas),axis=1)
        s=[]
        for k in [1,3,7]:
            # Averaged distances protect against a single coincidental close gas.
            a=dn[:,:min(k,len(normal))].mean(axis=1)
            b=dg[:,:min(k,len(gas))].mean(axis=1)
            s.append(a/(a+b+1e-9))
        out[v+'_neighbors']=np.mean(s,axis=0)
        s=[]
        covs=[]
        for z in [normal,gas]:
            cov=LedoitWolf().fit(z).covariance_ if len(z)>1 else np.eye(tr.shape[1])
            covs.append(cov)
        for shrink in [.3,.7,1.]:
            ll=[]
            for z,cov in zip([normal,gas],covs):
                cov=(1-shrink)*cov+shrink*np.diag(np.diag(cov))+np.eye(tr.shape[1])*1e-3
                delta=te-z.mean(axis=0)
                ll.append(-.5*(np.linalg.slogdet(cov)[1]+np.einsum('ij,jk,ik->i',delta,np.linalg.pinv(cov),delta)))
            s.append(expit(ll[1]-ll[0]))
        out[v+'_gaussian']=np.mean(s,axis=0)
        out[v+'_three_mean']=np.mean([out[v+'_'+m] for m in ['svm','neighbors','gaussian']],axis=0)
    memo[key]=out
    return out
records=[];audit=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b))
        ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q)
        inner={n:np.full(len(ref),np.nan) for n in outer}
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if not (~mask).any():continue
            p=predict(ref[~mask],ref[mask])
            for n in inner:inner[n][mask]=p[n]
        for name,score in outer.items():
            cal=inner[name];valid=np.isfinite(cal);norm=cal[valid&(y[ref]==0)]
            cut=float(np.quantile(norm,.99,method='higher')) if len(norm) else 1.
            # Also select threshold on INNER held-out groups only, bounded FP budget.
            chosen=cut;best=-1.
            for t in np.unique(np.r_[cal[valid],1.]):
                pred=cal[valid]>t; yy=y[ref][valid]
                fp=int(np.sum(pred&(yy==0)));tp=int(np.sum(pred&(yy==1)));fn=int(np.sum(~pred&(yy==1)))
                if fp>int(np.floor(.02*np.sum(yy==0))):continue
                f=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
                if f>best or (f==best and t>chosen):best=f;chosen=float(t)
            for rule,t in [('normal99',cut),('inner_f1_fp2pct',chosen)]:
                for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[k]),threshold=t,prediction=int(score[k]>t)))
                audit.append(dict(scheme=scheme,block=b,model=name,rule=rule,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum()),threshold=t,inner_best_F1=best))
        print('Finished',scheme,b,flush=True)
pp=pd.DataFrame(records)
base=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
base=base.loc[base.model.eq('max_union'),['scheme','TimeStamp','actual','score','predicted_train_fpr1']]
rows=[]
for (scheme,name,rule),g in pp.groupby(['scheme','model','rule'],sort=False):
    z=g.merge(base.drop(columns=['actual','score']),on=['scheme','TimeStamp'],validate='one_to_one')
    for combine in ['new_only','OR_RF','AND_RF']:
        pred=z.prediction.astype(bool)
        if combine=='OR_RF':pred=pred|z.predicted_train_fpr1
        if combine=='AND_RF':pred=pred&z.predicted_train_fpr1
        tn,fp,fn,tp=confusion_matrix(z.actual,pred,labels=[0,1]).ravel()
        rows.append(dict(scheme=scheme,model=name,rule=rule,combine=combine,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,AP=average_precision_score(z.actual,z.score),shots=len(z)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_supervised_geometry_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_supervised_geometry_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_supervised_geometry_fold_audit.csv',index=False,encoding='utf-8-sig')
for scheme in metrics.scheme.unique():
    print('\n',scheme)
    print(metrics.loc[metrics.scheme.eq(scheme)].sort_values(['F1','FP'],ascending=[False,True]).head(12).to_string(index=False))


## Executed results
Whole evaluation: best F1 does not exceed baseline 0.296 (TP4 FP6).
Forward evaluation: RF AND shake-SVM with inner-only threshold gives TP2 FP3 FN11 TN424 F1=0.222; whole evaluation TP3 FP5 F1=0.240. Exploratory only; no promotion.
Results and fold audit are in reports/tables/rg3_supervised_geometry_*.csv.